# **A/B Test: Spotify Campaign**

# Background Problem

**Problem:**
Tingkat konversi pembayaran (payment conversion rate) pada kelompok Control masih rendah (sebanyak 30,3% pengguna keluar tanpa melakukan pembayaran).

**Initiative:**
Mengevaluasi strategi kampanye pada kelompok Target untuk meningkatkan conversion rate dan mendorong peningkatan rata-rata durasi sesi (Duration Played).



# Identify Change and Hypothesis

**Identify Change:**
Kita ingin mengevaluasi dampak dari penerapan kampanye promosi berorientasi target (misalnya banner diskon yang dipersonalisasi dan rekomendasi konten yang disesuaikan) untuk pengguna paket Free agar terdorong beralih ke Premium.

* **Kondisi Saat Ini (Control):** Pengguna Free melihat iklan suara standar dan banner promosi umum tanpa penawaran khusus untuk pembaruan ke Premium.

* **Perubahan yang Diajukan (Target / Variant):** Pengguna Free menerima tampilan promosi khusus yang ditargetkan dengan penawaran diskon/uji coba yang dipersonalisasi serta rekomendasi konten yang relevan.

* **Perbedaan Utama:** Kelompok Target mendapatkan pesan penawaran yang dipersonalisasi dan elemen visual yang mencolok untuk mengurangi tingkat pengguna yang tidak melakukan pembayaran, sedangkan kelompok Control tetap melihat promosi standar yang sudah ada.

# Create a Hypothesis

Kami percaya **menerapkan kampanye promosi khusus dan rekomendasi personal** akan membuat **lebih banyak pengguna Free beralih ke Premium** karena **penawaran yang relevan dapat meningkatkan engagement dan mengurangi drop-off pengguna.**

# Define Metrics

## Success Metrics
**Payment Conversion Rate (CVR)**

* **Definisi:** Persentase pengguna yang melakukan pembayaran (Yes).

* **Alasan:** Jika kampanye promosi berorientasi target berhasil dan menarik bagi pengguna paket Free, mereka akan lebih terdorong untuk beralih dan membayar paket Premium.

**Hipotesis Statistik:**

$H_0: \text{CVR}_B \le \text{CVR}_A$

$H_1: \text{CVR}_B > \text{CVR}_A$

**Ekspektasi:** Kita berharap Conversion Rate dari perubahan yang baru (treatment / kelompok B) lebih bagus daripada kelompok control / A.

## Guardrail Metrics
**Average Session Duration (Duration Played)**

* **Definisi:** Rata-rata durasi waktu yang dihabiskan pengguna untuk mendengarkan konten di Spotify.

* **Alasan:** Kita ingin konversi pembayaran meningkat tanpa mengorbankan durasi engagement pengguna di aplikasi. Jangan sampai pengguna yang beralih ke Premium justru berkurang durasi mendengarnya.

# Success Metrics


In [29]:
import numpy as np
import pandas as pd
import os

pd.options.display.max_columns = 999
pd.options.display.float_format = "{:.2f}".format

# Read Data

In [30]:
df = pd.read_csv('/content/Assignment - spotify_campaign_data.csv')

df.head()

,Group,Engagement,Duration Played,Payment,Subscription
0,Control,"0,6496714153","122,9461844",Yes,Premium
1,Control,"0,5861735699","118,0767531",Yes,Free
2,Control,"0,6647688538","148,5537227",Yes,Premium
3,Control,"0,7523029856","165,9849342",No,Premium
4,Control,"0,5765846625","140,6054083",Yes,Free


In [31]:
df.value_counts('Payment', normalize=True)

,proportion
Payment,
Yes,0.75
No,0.25


# Success Metrics: Conversion Rate

Membandingkan conversion rate yang variant vs control tanpa uji statistika (langsung compare conversion rate)

In [32]:
# Control
df[df['Group'] == 'Control'].value_counts('Payment', normalize=True)

,proportion
Payment,
Yes,0.70
No,0.30


In [33]:
# Target
df[df['Group'] == 'Target'].value_counts('Payment', normalize=True)

,proportion
Payment,
Yes,0.80
No,0.20


Berdasarkan perbandingan langsung (tanpa uji statistik), kelompok Target (Variant) menghasilkan conversion rate yang lebih tinggi sebesar 80% (0.80) dibandingkan dengan kelompok Control yang memiliki conversion rate sebesar 70% (0.70). Hal ini menunjukkan adanya potensi peningkatan konversi sebesar 10% (atau 1000 bps) dari penerapan kampanye baru.

# Dengan Uji Statistik

Untuk membandingkan proporsi dari 2 kelompok sample (Control vs Variant), umumnya bisa menggunakan two-sample proportion z-test

Notasi uji hipotesis

$H_0: p_{target} \leq p_{control}$

$H_1: p_{target} > p_{control}$


Cara baca:

- Lihat di bagian $H_1$: Kita ingin menguji, apakah $p_{target}$ (conversion rate target) lebih tinggi dari conversion rate control
- Kalau p-value > 0.05: tolak $H_1$ karena tidak terbukti signifikan
- Kalau p-value < 0.05: tidak menolak $H_1$ karena terbukti signifikan

In [34]:
from statsmodels.stats.proportion import proportions_ztest

In [35]:
# 1. Konversi kolom 'Payment' menjadi angka (1 untuk 'Yes', 0 untuk 'No')
df['Payment_Num'] = (df['Payment'] == 'Yes').astype(int)

# 2. Bagi data berdasarkan Group (Control vs Target)
group_a = df[df['Group'] == 'Control']
group_b = df[df['Group'] == 'Target']

# 3. Hitung total konversi (jumlah nilai 1)
conversion = [group_b['Payment_Num'].sum(), group_a['Payment_Num'].sum()]
print("Jumlah user yang convert (Target, Control):", conversion)

# 4. Hitung total jumlah sampel masing-masing kelompok
sample = [len(group_b), len(group_a)]
print("Jumlah sample masing-masing kelompok (Target, Control):", sample)

# 5. Jalankan Two-Sample Proportion Z-Test (H1: Target > Control)
z_stat, p_value = proportions_ztest(count=conversion, nobs=sample, alternative='larger')

# 6. Tampilkan Hasil
print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# CVR (Conversion Rate)
cvr_control = group_a['Payment_Num'].mean()
cvr_target = group_b['Payment_Num'].mean()

print(f"CVR Control: {cvr_control:.3f}")
print(f"CVR Target: {cvr_target:.3f}")

# Keputusan Uji Hipotesis
if p_value < 0.05:
    print("\nKesimpulan: P-value < 0.05. Tolak H0. Terdapat peningkatan Conversion Rate yang signifikan pada kelompok Target.")
else:
    print("\nKesimpulan: P-value >= 0.05. Gagal tolak H0. Peningkatan Conversion Rate tidak signifikan secara statistik.")

Jumlah user yang convert (Target, Control): [np.int64(39907), np.int64(34844)]
Jumlah sample masing-masing kelompok (Target, Control): [50000, 50000]

Z-statistic: 36.8534
P-value: 0.0000
CVR Control: 0.697
CVR Target: 0.798

Kesimpulan: P-value < 0.05. Tolak H0. Terdapat peningkatan Conversion Rate yang signifikan pada kelompok Target.


Hasil uji Two-Sample Proportion Z-Test menghasilkan nilai p-value = 0.0000 (p-value < 0.05). Dengan demikian, kita menolak $H_0$ dan menyimpulkan bahwa $H_1$ terbukti secara signifikan. Penerapan kampanye baru pada kelompok Target terbukti mampu meningkatkan conversion rate secara signifikan (dari 69.7% pada kelompok Control menjadi 79.8% pada kelompok Target).

# Guardrail Metrics: Session Duration

Meskipun conversion rate bagus, kita juga ingin memastikan apakah rata-rata durasi orang membuka app meningkat atau lebih baik di kelompok variant daripada control.

In [27]:
# 1. Ubah koma menjadi titik dan konversi tipe data ke float
df['Duration Played'] = df['Duration Played'].astype(str).str.replace(',', '.').astype(float)

# 2. Ambil data per kelompok
session_a = df[df['Group'] == 'Control']['Duration Played']
session_b = df[df['Group'] == 'Target']['Duration Played']

Membandingkan rata-rata antara 2 angka numerik bisa menggunakan independent t-test.


$
H_0 : \bar{x}_{target} \leq \bar{x}_{control}
$

$
H_1 : \bar{x}_{target} > \bar{x}_{control}
$

Cara baca:
- $H_1$: melihat apakah avg session duration target > control

In [28]:
from scipy import stats
# 3. Jalankan Independent T-Test
t_stat, p_value = stats.ttest_ind(session_b, session_a, alternative='greater')

# 4. Tampilkan Hasil
print(f"P-value: {p_value:.4f}")
print(f"Mean of session duration Control (A): {session_a.mean():.2f}")
print(f"Mean of session duration Target (B): {session_b.mean():.2f}")

P-value: 0.0000
Mean of session duration Control (A): 120.07
Mean of session duration Target (B): 159.91


P-value = 0.0000 (p-value < 0.05), artinya $H_1$ terbukti secara signifikan — terdapat peningkatan rata-rata session duration (duration played) secara signifikan pada kelompok Target dibandingkan kelompok Control, yaitu naik dari 120,07 menit menjadi 159,91 menit.

# Summary

Hasil A/B testing menunjukkan bahwa eksperimen kampanye berorientasi target berhasil memberikan dampak positif yang signifikan pada seluruh metrik utama:

* **Peningkatan Conversion Rate (Success Metric):** Strategi kampanye baru berhasil menaikkan payment conversion rate dari 69,7% (kelompok Control) menjadi 79,8% (kelompok Target). Uji Two-Sample Proportion Z-Test membuktikan bahwa kenaikan sebesar 10,1% ini signifikan secara statistik ($p\text{-value} = 0,0000 < 0.05$).

* **Peningkatan Session Duration (Guardrail Metric):** Eksperimen tidak mengorbankan user engagement. Uji Independent T-Test menunjukkan adanya peningkatan rata-rata durasi mendengarkan (duration played) secara signifikan ($p\text{-value} = 0,0000 < 0.05$), yaitu naik dari 120,07 menit menjadi 159,91 menit.

**Rekomendasi Bisnis**

Karena kampanye baru terbukti secara signifikan dapat meningkatkan konversi pembayaran sekaligus memperlama durasi sesi pengguna tanpa dampak negatif, direkomendasikan untuk meluncurkan (roll out) strategi kampanye ini ke seluruh pengguna Free.